# 03. 최종 모델 비교

2단계에서 선정된 모델별 최적 variant를 고정하고 최종 성능을 비교한다.

| 모델 | 유형 | variant |
|------|------|---------|
| BGE (`bge-reranker-v2-m3`) | Cross-encoder | 2단계 선정 |
| GTE (`gte-multilingual-reranker-base`) | Cross-encoder | 2단계 선정 |
| Jina (`jina-reranker-v2-base-multilingual`) | Cross-encoder | 2단계 선정 |
| EXAONE (`EXAONE-3.5-7.8B-Instruct`) | LLM Listwise | - |
| CLOVA Reranker | API | 참고용 |

- **입력**: `data/gold_candidate_pool.json`, `data/results/best_variant_per_model.json`
- **출력**: `data/results/final/final_comparison.json`, 시각화 PNG

In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'

# CWD에서 위로 올라가며 evaluation/ 폴더가 있는 프로젝트 루트를 탐색
def find_root(marker: str = 'evaluation') -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / marker).is_dir():
            return p
        p = p.parent
    raise FileNotFoundError(f"프로젝트 루트 못 찾음 ('{marker}/' 없음)")

ROOT       = find_root('evaluation')          # .../somin (= librAIan)
RERANK_DIR = ROOT / 'experiments' / 'rerank'

sys.path.insert(0, str(RERANK_DIR / 'src'))

from book_text_variants import VARIANTS, VARIANT_DESC
from cross_encoder_reranker import CrossEncoderReranker, MODELS

# metrics.py를 파일 경로로 직접 로드
_metrics_path = ROOT / 'evaluation' / 'metrics' / 'metrics.py'
_spec = importlib.util.spec_from_file_location('metrics', _metrics_path)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
mrr_at_k  = _mod.mrr_at_k
ndcg_at_k = _mod.ndcg_at_k
hit_at_k  = _mod.hit_at_k

DATA_DIR   = RERANK_DIR / 'data'
RESULT_DIR = DATA_DIR / 'results' / 'final'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2
DEVICE = 'cuda'  # GPU 없으면 'cpu'

print(f'ROOT: {ROOT}')
print(f'CWD:  {Path.cwd()}')

## 1. 데이터 및 최적 variant 로드

In [ ]:
with open(DATA_DIR / 'gold_candidate_pool.json') as f:
    gold_pool = json.load(f)

with open(DATA_DIR / 'results' / 'best_variant_per_model.json') as f:
    best_variant = json.load(f)

print('모델별 최적 variant:')
for model, var in best_variant.items():
    print(f'  {model}: Variant {var} — {VARIANT_DESC[var]}')

## 2. Cross-encoder 3개 모델 평가

In [ ]:
def evaluate_model(reranker, gold_pool, text_fn, model_label):
    results = []
    for scenario in gold_pool:
        query = scenario['semantic_query']
        candidates = scenario['candidates']

        reranked = reranker.rerank(query, candidates, text_fn)
        reranked_isbns = [r['book']['isbn'] for r in reranked]

        relevant_isbns = [c['isbn'] for c in candidates if c['final_grade'] >= RELEVANT_THRESHOLD]
        relevance_scores = {c['isbn']: c['final_grade'] for c in candidates}

        results.append({
            'model': model_label,
            'scenario_id': scenario['scenario_id'],
            'n_relevant': len(relevant_isbns),
            'mrr@10':  mrr_at_k(relevant_isbns, reranked_isbns, k=10),
            'ndcg@10': ndcg_at_k(relevance_scores, reranked_isbns, k=10),
            'hit@10':  hit_at_k(relevant_isbns, reranked_isbns, k=10),
            'mrr@5':   mrr_at_k(relevant_isbns, reranked_isbns, k=5),
        })
    return results


all_results = []

for model_name, model_id in MODELS.items():
    var = best_variant.get(model_name, 'A')
    print(f'\n── {model_name} (variant {var}) ──')
    reranker = CrossEncoderReranker(model_id, device=DEVICE)
    results = evaluate_model(reranker, gold_pool, VARIANTS[var], model_name)
    all_results.extend(results)

    avg_mrr = sum(r['mrr@10'] for r in results if r['n_relevant'] > 0) / max(1, sum(1 for r in results if r['n_relevant'] > 0))
    print(f'  avg MRR@10 = {avg_mrr:.4f}')
    del reranker

print('\nCross-encoder 평가 완료.')

## 3. LLM Listwise: EXAONE

> `llm_listwise_reranker.py` 구현 후 아래 셀을 채운다.

In [ ]:
# TODO: EXAONE listwise reranker 구현 후 활성화
# from llm_listwise_reranker import ExaoneListwiseReranker
#
# exaone = ExaoneListwiseReranker(
#     model_name='LG-AI-Research/EXAONE-3.5-7.8B-Instruct',
#     device=DEVICE,
# )
# results = evaluate_model(exaone, gold_pool, text_fn=None, model_label='EXAONE')
# all_results.extend(results)

print('[EXAONE] 구현 대기 중 — llm_listwise_reranker.py 완성 후 활성화')

## 4. CLOVA Reranker (참고용)

> 기존 `08_eval_reranking_comparison.ipynb` 결과 또는 API 직접 호출.

In [ ]:
# TODO: CLOVA API 결과 로드 또는 호출
# 기존 08 노트북 결과를 JSON으로 저장 후 여기서 로드하는 방식 권장

print('[CLOVA] 기존 08_eval_reranking_comparison.ipynb 결과를 참고')

## 5. 최종 비교표 및 시각화

In [ ]:
df = pd.DataFrame(all_results)

# relevant 시나리오만 집계
df_rel = df[df['n_relevant'] > 0]

summary = (
    df_rel.groupby('model')[['mrr@10', 'ndcg@10', 'hit@10', 'mrr@5']]
    .mean()
    .sort_values('mrr@10', ascending=False)
    .round(4)
)
print('=== 최종 모델 비교 (MRR@10 내림차순) ===')
print(summary.to_string())

In [ ]:
# 바 차트
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['mrr@10', 'ndcg@10', 'hit@10']
titles  = ['MRR@10', 'NDCG@10', 'Hit@10']

for ax, metric, title in zip(axes, metrics, titles):
    summary[metric].plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('최종 Reranker 모델 비교', fontsize=13)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 시나리오 타입별 성능 분해
# scenario_data_after_retrieval.json에서 query_type 로드
with open(Path('../../../evaluation/dataset/scenario_data_after_retrieval.json')) as f:
    scenario_data = json.load(f)

qtype_map = {s['scenario_id']: s['query_type'] for s in scenario_data}
df['query_type'] = df['scenario_id'].map(qtype_map)

type_summary = (
    df[df['n_relevant'] > 0]
    .groupby(['query_type', 'model'])['mrr@10']
    .mean()
    .unstack('model')
    .round(4)
)
print('=== query_type별 MRR@10 ===')
print(type_summary.to_string())

In [ ]:
# 결과 저장
final_output = {
    'summary': summary.reset_index().to_dict(orient='records'),
    'by_query_type': type_summary.reset_index().to_dict(orient='records'),
    'raw': df[['model', 'scenario_id', 'query_type', 'mrr@10', 'ndcg@10', 'hit@10']].to_dict(orient='records'),
}
with open(RESULT_DIR / 'final_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

print(f'저장 완료: {RESULT_DIR / "final_comparison.json"}')
print(f'\n최적 모델: {summary.index[0]}  (MRR@10 = {summary["mrr@10"].iloc[0]:.4f})')